IMPORTING ALL THE REQUIRED IMPORTS

In [1]:
import pandas as pd
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,ExtraTreesRegressor, AdaBoostRegressor, BaggingRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, HuberRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'pandas'

Checking model imports

In [ ]:
def rmse(y_true, y_pred):
    """Sklearn 1.8 dropped the squared param — use this helper instead."""
    return np.sqrt(mean_squared_error(y_true, y_pred))
 
try:
    from xgboost import XGBRegressor;        XGBOOST_OK  = True;  print("XGBoost  : loaded")
except ImportError:                          XGBOOST_OK  = False; print("XGBoost  : pip install xgboost")
try:
    from lightgbm import LGBMRegressor;      LGBM_OK     = True;  print("LightGBM : loaded")
except ImportError:                          LGBM_OK     = False; print("LightGBM : pip install lightgbm")
try:
    from catboost import CatBoostRegressor;  CATBOOST_OK = True;  print("CatBoost : loaded")
except ImportError:                          CATBOOST_OK = False; print("CatBoost : pip install catboost")  

XGBoost  : loaded
LightGBM : loaded
CatBoost : loaded


LOADING THE DATASET

In [ ]:
df = pd.read_csv('train_final.csv')
print(f'\nLoaded: {df.shape[0]} rows × {df.shape[1]} cols')


Loaded: 51645 rows × 44 cols


BUILDING MODELS 

In [ ]:
ride_df = df.groupby('ride_id').agg(
    num_passengers     = ('ride_id',           'count'),
    travel_from        = ('travel_from',        'first'),
    max_capacity       = ('max_capacity',        'first'),
    month              = ('month',              'first'),
    day_of_week        = ('day_of_week',         'first'),
    week               = ('week',               'first'),
    hour               = ('hour',               'first'),
    minute             = ('minute',             'first'),
    is_bus             = ('is_bus',             'first'),
    is_weekend         = ('is_weekend',          'first'),
    mpesa_ratio        = ('is_mpesa',            'mean'),
    avg_seat_row       = ('seat_row',            'mean'),
    fuel_price_kes     = ('fuel_price_kes',      'first'),
    fare_kes           = ('fare_kes',            'mean'),
    distance_km        = ('distance_km',         'first'),
    is_holiday         = ('is_holiday',          'first'),
    is_holiday_eve     = ('is_holiday_eve',      'first'),
    is_holiday_aft     = ('is_holiday_aft',      'first'),
    days_to_holiday    = ('days_to_holiday',     'first'),
    is_school_hol      = ('is_school_hol',       'first'),
    is_rainy_season    = ('is_rainy_season',     'first'),
    is_long_rains      = ('is_long_rains',       'first'),
    day_of_month       = ('day_of_month',        'first'),
    is_month_end       = ('is_month_end',        'first'),
    is_month_start     = ('is_month_start',      'first'),
    is_morning_peak    = ('is_morning_peak',     'first'),
    is_evening_peak    = ('is_evening_peak',     'first'),
    n_competitors      = ('n_competitors',       'first'),
    origin_pop_proxy   = ('origin_pop_proxy',    'first'),
    avg_ticket_lead    = ('ticket_lead_days',    'mean'),
    min_ticket_lead    = ('ticket_lead_days',    'min'),
    max_ticket_lead    = ('ticket_lead_days',    'max'),
    std_ticket_lead    = ('ticket_lead_days',    'std'),
    avg_booking_hour   = ('booking_hour',        'mean'),
    avg_booking_dow    = ('booking_dow',         'mean'),
    n_weekend_bookings = ('booked_on_weekend',   'sum'),
    avg_lead_cap_ratio = ('lead_capacity_ratio', 'mean'),
).reset_index()
 
ride_df['std_ticket_lead'] = ride_df['std_ticket_lead'].fillna(0)
ride_df['quarter']         = ((ride_df['month'] - 1) // 3) + 1
ride_df['time_float']      = ride_df['hour'] + ride_df['minute'] / 60.0
ride_df['slot']            = (ride_df['travel_from'] + '_' +
                               ride_df['hour'].astype(str) + '_' +
                               ride_df['minute'].astype(str))
ride_df['route']           = (ride_df['travel_from'] + '_' +
                               ride_df['is_bus'].map({1:'bus', 0:'shuttle'}))
ride_df['weekend_holiday'] = ride_df['is_weekend'] * (ride_df['is_holiday'] + ride_df['is_school_hol'])
ride_df['lead_spread']     = ride_df['max_ticket_lead'] - ride_df['min_ticket_lead']
ride_df['booking_urgency'] = ride_df['avg_ticket_lead'] / 21.0
 
for col, src in [('travel_from_enc','travel_from'),('slot_enc','slot'),('route_enc','route')]:
    le = LabelEncoder(); ride_df[col] = le.fit_transform(ride_df[src])
 
print(f'Ride-level dataset: {len(ride_df)} rides')
 
 
#TRAIN/TEST SPLIT
BASE_FEATURES = [
    'max_capacity','month','day_of_week','week','hour','minute',
    'is_bus','is_weekend','travel_from_enc','slot_enc','route_enc',
    'day_of_month','quarter','avg_seat_row','mpesa_ratio','time_float',
    'fuel_price_kes','fare_kes','distance_km','is_holiday','is_holiday_eve',
    'is_holiday_aft','is_school_hol','days_to_holiday','is_rainy_season',
    'is_long_rains','is_month_end','is_month_start','is_morning_peak',
    'is_evening_peak','n_competitors','origin_pop_proxy','weekend_holiday',
    'avg_ticket_lead','min_ticket_lead','max_ticket_lead','std_ticket_lead',
    'avg_booking_hour','avg_booking_dow','n_weekend_bookings',
    'avg_lead_cap_ratio','lead_spread','booking_urgency',
]
 
X = ride_df[BASE_FEATURES].copy()
y = ride_df['num_passengers']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
 
 
#KFOLD SMOOTHED MEAN ENCODING
global_mean = y_train.mean()
smoothing   = 8
kf          = KFold(n_splits=5, shuffle=True, random_state=42)
 
def kfold_encode(col, grp, is_str=False):
    grp_tr = ride_df.loc[X_train.index, grp].values if is_str else X_train[grp].values
    grp_te = ride_df.loc[X_test.index,  grp].values if is_str else X_test[grp].values
    oof    = np.zeros(len(X_train))
    for tr, val in kf.split(X_train):
        c  = pd.Series(y_train.values[tr]).groupby(grp_tr[tr]).count()
        m  = pd.Series(y_train.values[tr]).groupby(grp_tr[tr]).mean()
        sm = (m * c + global_mean * smoothing) / (c + smoothing)
        oof[val] = pd.Series(grp_tr[val]).map(sm).fillna(global_mean).values
    fc  = pd.Series(y_train.values).groupby(grp_tr).count()
    fm  = pd.Series(y_train.values).groupby(grp_tr).mean()
    fsm = (fm * fc + global_mean * smoothing) / (fc + smoothing)
    X_train[col] = oof
    X_test[col]  = pd.Series(grp_te).map(fsm).fillna(global_mean).values
 
for col, grp, is_str in [
    ('slot_mean','slot',True),('route_mean','route',True),('origin_mean','travel_from',True),
    ('dow_mean','day_of_week',False),('month_mean','month',False),('hour_mean','hour',False),
    ('week_mean','week',False),('quarter_mean','quarter',False),('dom_mean','day_of_month',False),
]:
    kfold_encode(col, grp, is_str)
 
 
#INTERACTION FEATURES
for split in [X_train, X_test]:
    split['slot_x_week']      = split['slot_mean']       * split['week_mean']
    split['slot_x_dow']       = split['slot_mean']       * split['dow_mean']
    split['slot_x_month']     = split['slot_mean']       * split['month_mean']
    split['slot_x_dom']       = split['slot_mean']       * split['dom_mean']
    split['route_x_month']    = split['route_mean']      * split['month_mean']
    split['route_x_week']     = split['route_mean']      * split['week_mean']
    split['origin_x_slot']    = split['origin_mean']     * split['slot_mean']
    split['holiday_x_slot']   = split['is_holiday']      * split['slot_mean']
    split['school_x_slot']    = split['is_school_hol']   * split['slot_mean']
    split['fuel_x_dist']      = split['fuel_price_kes']  * split['distance_km']
    split['comp_x_origin']    = split['n_competitors']   * split['origin_mean']
    split['rain_x_slot']      = split['is_rainy_season'] * split['slot_mean']
    split['fare_x_dist']      = split['fare_kes']        / split['distance_km']
    split['capacity_x_bus']   = split['max_capacity']    * split['is_bus']
    split['lead_x_slot']      = split['avg_ticket_lead'] * split['slot_mean']
    split['lead_x_week']      = split['avg_ticket_lead'] * split['week_mean']
    split['lead_x_holiday']   = split['avg_ticket_lead'] * split['is_holiday']
    split['lead_x_school']    = split['avg_ticket_lead'] * split['is_school_hol']
    split['spread_x_slot']    = split['lead_spread']     * split['slot_mean']
    split['urgency_x_origin'] = split['booking_urgency'] * split['origin_mean']
    split['minlead_x_slot']   = split['min_ticket_lead'] * split['slot_mean']
 
FINAL_FEATURES = X_train.columns.tolist()
print(f'Total features: {len(FINAL_FEATURES)}\n')
 
 
#EPOCH PRINTING HELPER
PRINT_EPOCHS = True        # set False to silence per-epoch output
EPOCH_INTERVAL = 100       # print every N epochs for boosting models
RF_CHECKPOINTS = [50, 100, 200, 300, 400, 500]   # tree counts to report for RF/ET/Bagging
 
def print_epoch(name, epoch, n_total, tr_rmse, va_rmse, va_r2, best=False):
    if not PRINT_EPOCHS:
        return
    tag = ' ← best' if best else ''
    print(f'  [{name}] epoch {epoch:>4}/{n_total}  '
          f'train_rmse={tr_rmse:.4f}  val_rmse={va_rmse:.4f}  val_R²={va_r2:.4f}{tag}')
 
 
#DEFINE MODELS
CATEGORIES = {
    'Linear Regression':'Linear',    'Ridge':        'Linear',
    'Lasso':            'Linear',    'ElasticNet':   'Linear',
    'Huber':            'Linear',    'KNN':          'Distance',
    'SVR':              'Distance',  'Decision Tree':'Tree',
    'Bagging':          'Ensemble',  'Extra Trees':  'Ensemble',
    'AdaBoost':         'Boosting',  'Random Forest':'Ensemble',
    'GBM':              'Boosting',  'XGBoost':      'Boosting',
    'LightGBM':         'Boosting',  'CatBoost':     'Boosting',
}
 
CAT_HEX = {
    'Linear':'#2171b5', 'Distance':'#6a3d9a', 'Tree':'#33a02c',
    'Ensemble':'#b15928', 'Boosting':'#e31a1c',
}
 
 
#TRAIN & EVALUATE
print('=' * 68)
print(f'{"Model":<22} {"R²":>8} {"MAE":>8} {"RMSE":>8}  {"Epochs/Trees":>14}')
print('=' * 68)
 
results_metrics = {}
results_preds   = {}
epoch_log       = {}
 
#helper: fit RF-family with warm_start checkpoints
def fit_rf_with_epochs(model, name, checkpoints):
    """Fits model using warm_start, printing val R² at each checkpoint."""
    ep_va_r2 = []; ep_va_rmse = []; ep_tr_rmse = []
    model.warm_start = True
    best_r2 = -np.inf; best_n = checkpoints[0]
    for n in checkpoints:
        model.n_estimators = n
        model.fit(X_train[FINAL_FEATURES], y_train)
        tr_r = rmse(y_train, model.predict(X_train[FINAL_FEATURES]))
        va_p = model.predict(X_test[FINAL_FEATURES])
        va_r = rmse(y_test, va_p)
        va_r2 = r2_score(y_test, va_p)
        ep_tr_rmse.append(tr_r); ep_va_rmse.append(va_r); ep_va_r2.append(va_r2)
        is_best = va_r2 > best_r2
        if is_best: best_r2 = va_r2; best_n = n
        print_epoch(name, n, checkpoints[-1], tr_r, va_r, va_r2, best=is_best)
    model.warm_start = False
    return ep_tr_rmse, ep_va_rmse, ep_va_r2, best_n
 
#1. Linear models (no epochs)
for name, model in [
    ('Linear Regression', LinearRegression()),
    ('Ridge',             Ridge(alpha=1.0)),
    ('Lasso',             Lasso(alpha=0.1, max_iter=1000)),
    ('ElasticNet',        ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=1000)),
    ('Huber',             HuberRegressor(max_iter=300)),
]:
    model.fit(X_train[FINAL_FEATURES], y_train)
    preds = model.predict(X_test[FINAL_FEATURES])
    r2    = r2_score(y_test, preds)
    mae   = mean_absolute_error(y_test, preds)
    rm    = rmse(y_test, preds)
    n_itr = getattr(model, 'n_iter_', getattr(model, 'max_iter', '—'))
    results_metrics[name] = {'r2':r2,'mae':mae,'rmse':rm,'cat':CATEGORIES[name]}
    results_preds[name]   = preds
    print(f'{name:<22} {r2:>8.4f} {mae:>8.3f} {rm:>8.3f}  {str(n_itr):>14}')
 
#2. KNN, SVR (no epochs)
for name, model in [
    ('KNN', KNeighborsRegressor(n_neighbors=10)),
    ('SVR', SVR(kernel='rbf', C=10, epsilon=0.5)),
]:
    model.fit(X_train[FINAL_FEATURES], y_train)
    preds = model.predict(X_test[FINAL_FEATURES])
    r2 = r2_score(y_test, preds); mae = mean_absolute_error(y_test, preds); rm = rmse(y_test, preds)
    results_metrics[name] = {'r2':r2,'mae':mae,'rmse':rm,'cat':CATEGORIES[name]}
    results_preds[name]   = preds
    print(f'{name:<22} {r2:>8.4f} {mae:>8.3f} {rm:>8.3f}  {"—":>14}')
 
#3. Decision Tree (no epochs)
name = 'Decision Tree'
model = DecisionTreeRegressor(max_depth=10, random_state=42)
model.fit(X_train[FINAL_FEATURES], y_train)
preds = model.predict(X_test[FINAL_FEATURES])
r2 = r2_score(y_test, preds); mae = mean_absolute_error(y_test, preds); rm = rmse(y_test, preds)
results_metrics[name] = {'r2':r2,'mae':mae,'rmse':rm,'cat':CATEGORIES[name]}
results_preds[name]   = preds
print(f'{name:<22} {r2:>8.4f} {mae:>8.3f} {rm:>8.3f}  {"—":>14}')
 
#4. AdaBoost — per-epoch via staged_predict
name = 'AdaBoost'
n_est = 200
print(f'\n[{name}] training {n_est} boosting rounds...')
model = AdaBoostRegressor(n_estimators=n_est, learning_rate=0.05, random_state=42)
model.fit(X_train[FINAL_FEATURES], y_train)
 
ada_tr, ada_va, ada_r2_list = [], [], []
best_r2 = -np.inf; best_ep = 1
for ep, (pred_tr, pred_va) in enumerate(zip(
        model.staged_predict(X_train[FINAL_FEATURES]),
        model.staged_predict(X_test[FINAL_FEATURES])), 1):
    tr_r = rmse(y_train, pred_tr); va_r = rmse(y_test, pred_va)
    va_r2 = r2_score(y_test, pred_va)
    ada_tr.append(tr_r); ada_va.append(va_r); ada_r2_list.append(va_r2)
    is_best = va_r2 > best_r2
    if is_best: best_r2 = va_r2; best_ep = ep
    if ep % EPOCH_INTERVAL == 0 or ep == 1 or ep == n_est or is_best:
        print_epoch(name, ep, n_est, tr_r, va_r, va_r2, best=is_best)
 
epoch_log[name] = {'train': ada_tr, 'val': ada_va, 'val_r2': ada_r2_list}
preds = model.predict(X_test[FINAL_FEATURES])
r2 = r2_score(y_test, preds); mae = mean_absolute_error(y_test, preds); rm = rmse(y_test, preds)
results_metrics[name] = {'r2':r2,'mae':mae,'rmse':rm,'cat':CATEGORIES[name]}
results_preds[name]   = preds
print(f'\n{name:<22} {r2:>8.4f} {mae:>8.3f} {rm:>8.3f}  {n_est:>14} rounds\n')
 
#5. Bagging — warm_start checkpoints
name = 'Bagging'
checkpoints = [10, 25, 50, 75, 100]
print(f'[{name}] training with warm_start checkpoints: {checkpoints}')
model = BaggingRegressor(n_estimators=1, warm_start=True, random_state=42)
tr_r_list, va_r_list, va_r2_list, best_n = fit_rf_with_epochs(model, name, checkpoints)
epoch_log[name] = {'checkpoints': checkpoints, 'train': tr_r_list, 'val': va_r_list, 'val_r2': va_r2_list}
preds = model.predict(X_test[FINAL_FEATURES])
r2 = r2_score(y_test, preds); mae = mean_absolute_error(y_test, preds); rm = rmse(y_test, preds)
results_metrics[name] = {'r2':r2,'mae':mae,'rmse':rm,'cat':CATEGORIES[name]}
results_preds[name]   = preds
print(f'\n{name:<22} {r2:>8.4f} {mae:>8.3f} {rm:>8.3f}  {checkpoints[-1]:>14} trees\n')
 
#6. Extra Trees — warm_start checkpoints
name = 'Extra Trees'
checkpoints = [25, 50, 100, 150, 200, 250, 300]
print(f'[{name}] training with warm_start checkpoints: {checkpoints}')
model = ExtraTreesRegressor(n_estimators=1, warm_start=True, max_depth=15, n_jobs=-1, random_state=42)
tr_r_list, va_r_list, va_r2_list, best_n = fit_rf_with_epochs(model, name, checkpoints)
epoch_log[name] = {'checkpoints': checkpoints, 'train': tr_r_list, 'val': va_r_list, 'val_r2': va_r2_list}
preds = model.predict(X_test[FINAL_FEATURES])
r2 = r2_score(y_test, preds); mae = mean_absolute_error(y_test, preds); rm = rmse(y_test, preds)
results_metrics[name] = {'r2':r2,'mae':mae,'rmse':rm,'cat':CATEGORIES[name]}
results_preds[name]   = preds
print(f'\n{name:<22} {r2:>8.4f} {mae:>8.3f} {rm:>8.3f}  {checkpoints[-1]:>14} trees\n')
 
#7. Random Forest — warm_start checkpoints
name = 'Random Forest'
checkpoints = [50, 100, 200, 300, 400, 500]
print(f'[{name}] training with warm_start checkpoints: {checkpoints}')
model = RandomForestRegressor(n_estimators=1, warm_start=True, max_depth=20,
                               min_samples_leaf=1, n_jobs=-1, random_state=42)
tr_r_list, va_r_list, va_r2_list, best_n = fit_rf_with_epochs(model, name, checkpoints)
epoch_log[name] = {'checkpoints': checkpoints, 'train': tr_r_list, 'val': va_r_list, 'val_r2': va_r2_list}
preds = model.predict(X_test[FINAL_FEATURES])
r2 = r2_score(y_test, preds); mae = mean_absolute_error(y_test, preds); rm = rmse(y_test, preds)
results_metrics[name] = {'r2':r2,'mae':mae,'rmse':rm,'cat':CATEGORIES[name]}
results_preds[name]   = preds
print(f'\n{name:<22} {r2:>8.4f} {mae:>8.3f} {rm:>8.3f}  {checkpoints[-1]:>14} trees\n')
 
#8. GBM — staged_predict gives every epoch
name = 'GBM'
n_est = 900
print(f'[{name}] training {n_est} boosting rounds...')
model = GradientBoostingRegressor(n_estimators=n_est, learning_rate=0.012,
                                   max_depth=6, subsample=0.8, random_state=42)
model.fit(X_train[FINAL_FEATURES], y_train)
 
gbm_tr, gbm_va, gbm_r2_list = [], [], []
best_r2 = -np.inf; best_ep = 1
for ep, (pred_tr, pred_va) in enumerate(zip(
        model.staged_predict(X_train[FINAL_FEATURES]),
        model.staged_predict(X_test[FINAL_FEATURES])), 1):
    tr_r  = rmse(y_train, pred_tr)
    va_r  = rmse(y_test, pred_va)
    va_r2 = r2_score(y_test, pred_va)
    gbm_tr.append(tr_r); gbm_va.append(va_r); gbm_r2_list.append(va_r2)
    is_best = va_r2 > best_r2
    if is_best: best_r2 = va_r2; best_ep = ep
    if ep % EPOCH_INTERVAL == 0 or ep == 1 or ep == n_est or is_best:
        print_epoch(name, ep, n_est, tr_r, va_r, va_r2, best=is_best)
 
epoch_log[name] = {'train': gbm_tr, 'val': gbm_va, 'val_r2': gbm_r2_list}
preds = model.predict(X_test[FINAL_FEATURES])
r2 = r2_score(y_test, preds); mae = mean_absolute_error(y_test, preds); rm = rmse(y_test, preds)
results_metrics[name] = {'r2':r2,'mae':mae,'rmse':rm,'cat':CATEGORIES[name]}
results_preds[name]   = preds
print(f'\n{name:<22} {r2:>8.4f} {mae:>8.3f} {rm:>8.3f}  {n_est:>14} rounds\n')
 
#9. XGBoost — evals_result() per round
if XGBOOST_OK:
    name = 'XGBoost'
    n_est = 2000
    print(f'[{name}] training up to {n_est} rounds (early stop=100)...')
    model = XGBRegressor(
        n_estimators=n_est, learning_rate=0.008, max_depth=7,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=2,
        reg_alpha=0.03, reg_lambda=1.0, early_stopping_rounds=100,
        eval_metric='rmse', n_jobs=-1, random_state=42, verbosity=0,
    )
    model.fit(X_train[FINAL_FEATURES], y_train,
              eval_set=[(X_train[FINAL_FEATURES], y_train),
                        (X_test[FINAL_FEATURES],  y_test)],
              verbose=False)
    try:
        hist       = model.evals_result()
        xgb_tr     = hist['validation_0']['rmse']
        xgb_va     = hist['validation_1']['rmse']
        best_r2_xgb = -np.inf; best_ep_xgb = 1
        for ep, (tr_r, va_r) in enumerate(zip(xgb_tr, xgb_va), 1):
            preds_ep = model.predict(X_test[FINAL_FEATURES], iteration_range=(0, ep))
            va_r2    = r2_score(y_test, preds_ep)
            is_best  = va_r2 > best_r2_xgb
            if is_best: best_r2_xgb = va_r2; best_ep_xgb = ep
            if ep % EPOCH_INTERVAL == 0 or ep == 1 or is_best:
                print_epoch(name, ep, len(xgb_tr), tr_r, va_r, va_r2, best=is_best)
        epoch_log[name] = {'train': xgb_tr, 'val': xgb_va}
    except Exception as e:
        print(f'  Epoch log error: {e}')
    preds = model.predict(X_test[FINAL_FEATURES])
    r2 = r2_score(y_test, preds); mae = mean_absolute_error(y_test, preds); rm = rmse(y_test, preds)
    best_n = getattr(model, 'best_iteration', n_est)
    results_metrics[name] = {'r2':r2,'mae':mae,'rmse':rm,'cat':CATEGORIES[name]}
    results_preds[name]   = preds
    print(f'\n{name:<22} {r2:>8.4f} {mae:>8.3f} {rm:>8.3f}  {best_n:>14} rounds\n')
 
#10. LightGBM — record_evaluation callback
if LGBM_OK:
    from lightgbm import early_stopping, record_evaluation
    name = 'LightGBM'
    n_est = 2000
    print(f'[{name}] training up to {n_est} rounds (early stop=100)...')
    model = LGBMRegressor(
        n_estimators=n_est, learning_rate=0.008, num_leaves=127,
        max_depth=10, subsample=0.8, colsample_bytree=0.8,
        min_child_samples=3, reg_alpha=0.03, reg_lambda=1.0,
        n_jobs=-1, random_state=42, verbose=-1,
    )
    cb_dict = {}
    model.fit(
        X_train[FINAL_FEATURES], y_train,
        eval_set=[(X_train[FINAL_FEATURES], y_train),
                  (X_test[FINAL_FEATURES],  y_test)],
        eval_names=['train', 'val'],
        callbacks=[
            early_stopping(stopping_rounds=100, verbose=False),
            record_evaluation(cb_dict),
        ],
    )
    try:
        lgbm_tr  = [v**0.5 for v in cb_dict['train']['l2']]  # l2 = MSE → sqrt = RMSE
        lgbm_va  = [v**0.5 for v in cb_dict['val']['l2']]
        best_r2_lgbm = -np.inf; best_ep_lgbm = 1
        preds_all_ep = []
        for ep, (tr_r, va_r) in enumerate(zip(lgbm_tr, lgbm_va), 1):
            preds_ep = model.predict(X_test[FINAL_FEATURES], num_iteration=ep)
            va_r2    = r2_score(y_test, preds_ep)
            is_best  = va_r2 > best_r2_lgbm
            if is_best: best_r2_lgbm = va_r2; best_ep_lgbm = ep
            if ep % EPOCH_INTERVAL == 0 or ep == 1 or is_best:
                print_epoch(name, ep, len(lgbm_tr), tr_r, va_r, va_r2, best=is_best)
        epoch_log[name] = {'train': lgbm_tr, 'val': lgbm_va}
    except Exception as e:
        print(f'  Epoch log error: {e}')
    preds = model.predict(X_test[FINAL_FEATURES])
    r2 = r2_score(y_test, preds); mae = mean_absolute_error(y_test, preds); rm = rmse(y_test, preds)
    best_n = getattr(model, 'best_iteration_', n_est)
    results_metrics[name] = {'r2':r2,'mae':mae,'rmse':rm,'cat':CATEGORIES[name]}
    results_preds[name]   = preds
    print(f'\n{name:<22} {r2:>8.4f} {mae:>8.3f} {rm:>8.3f}  {best_n:>14} rounds\n')
 
#11. CatBoost — get_evals_result() per iteration
if CATBOOST_OK:
    name = 'CatBoost'
    n_est = 2000
    print(f'[{name}] training up to {n_est} iterations (early stop=100)...')
    model = CatBoostRegressor(
        iterations=n_est, learning_rate=0.008, depth=8,
        l2_leaf_reg=2.0, early_stopping_rounds=100,
        eval_metric='RMSE', random_seed=42, verbose=0,
    )
    model.fit(X_train[FINAL_FEATURES], y_train,
              eval_set=(X_test[FINAL_FEATURES], y_test), verbose=False)
    try:
        hist    = model.get_evals_result()
        cb_tr   = hist.get('learn',      {}).get('RMSE', [])
        cb_va   = hist.get('validation', {}).get('RMSE', [])
        # CatBoost ntree_end must be <= best_iteration_ (0-based cap)
        best_iter  = getattr(model, 'best_iteration_', len(cb_tr) - 1)
        best_r2_cb = -np.inf; best_ep_cb = 1
        for ep, (tr_r, va_r) in enumerate(zip(cb_tr, cb_va), 1):
            safe_end = min(ep, best_iter)
            preds_ep = model.predict(X_test[FINAL_FEATURES], ntree_end=safe_end)
            va_r2    = r2_score(y_test, preds_ep)
            is_best  = va_r2 > best_r2_cb
            if is_best: best_r2_cb = va_r2; best_ep_cb = ep
            if ep % EPOCH_INTERVAL == 0 or ep == 1 or is_best:
                print_epoch(name, ep, len(cb_tr), tr_r, va_r, va_r2, best=is_best)
        epoch_log[name] = {'train': cb_tr, 'val': cb_va}
    except Exception as e:
        print(f'  Epoch log error: {e}')
    preds = model.predict(X_test[FINAL_FEATURES])
    r2 = r2_score(y_test, preds); mae = mean_absolute_error(y_test, preds); rm = rmse(y_test, preds)
    best_n = getattr(model, 'best_iteration_', n_est)
    results_metrics[name] = {'r2':r2,'mae':mae,'rmse':rm,'cat':CATEGORIES[name]}
    results_preds[name]   = preds
    print(f'\n{name:<22} {r2:>8.4f} {mae:>8.3f} {rm:>8.3f}  {best_n:>14} iters\n')
 
#FINAL SUMMARY
print('=' * 68)
print(f'{"Model":<22} {"R²":>8} {"MAE":>8} {"RMSE":>8}')
print('=' * 68)
for name, m in results_metrics.items():
    print(f'{name:<22} {m["r2"]:>8.4f} {m["mae"]:>8.3f} {m["rmse"]:>8.3f}')
print('=' * 68)
 
print('\n--- Ranking by R² ---')
for rank, (name, m) in enumerate(sorted(results_metrics.items(), key=lambda x: -x[1]['r2']), 1):
    bar = '█' * int(m['r2'] * 30)
    print(f'{rank:>2}. {name:<22} {m["r2"]:.4f}  {bar}')

Ride-level dataset: 6249 rides
Total features: 73

Model                        R²      MAE     RMSE    Epochs/Trees
Linear Regression        0.9280    1.683    2.418               —
Ridge                    0.9279    1.685    2.419            None
Lasso                    0.9223    1.705    2.512            1000
ElasticNet               0.9237    1.700    2.490            1000
Huber                    0.8258    2.353    3.761             300
KNN                      0.7839    2.723    4.190               —
SVR                      0.0852    5.600    8.619               —
Decision Tree            0.9038    1.433    2.795               —

[AdaBoost] training 200 boosting rounds...
  [AdaBoost] epoch    1/200  train_rmse=3.7507  val_rmse=4.1169  val_R²=0.7913 ← best
  [AdaBoost] epoch    3/200  train_rmse=3.4914  val_rmse=3.6441  val_R²=0.8365 ← best
  [AdaBoost] epoch    6/200  train_rmse=3.4542  val_rmse=3.6352  val_R²=0.8373 ← best
  [AdaBoost] epoch   11/200  train_rmse=3.3555  val_r

PLOTTING PREDICTION VS ACTUAL PLOT

In [ ]:
n_models = len(results_preds)
ncols    = 4
nrows    = int(np.ceil(n_models / ncols))
pmax     = max(y_test.values.max(), max(p.max() for p in results_preds.values())) + 2
 
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 4.5))
fig.patch.set_facecolor('#0f1117')
axes_flat = axes.flatten()
 
for i, (name, preds) in enumerate(results_preds.items()):
    ax  = axes_flat[i]
    cat = results_metrics[name]['cat']
    col = CAT_HEX.get(cat, '#888888')
    r2  = results_metrics[name]['r2']
    mae = results_metrics[name]['mae']
 
    ax.set_facecolor('#1a1d27')
    for sp in ax.spines.values(): sp.set_edgecolor('#2e3250')
    ax.hexbin(y_test.values, preds, gridsize=25, cmap='YlOrRd',
              mincnt=1, linewidths=0.2, extent=[0, pmax, 0, pmax])
    ax.plot([0, pmax], [0, pmax], color='#4fc3f7', lw=1.5, ls='--', alpha=0.9)
    ax.axhspan(pmax * 0.96, pmax, color=col, alpha=0.6)
    ax.set_xlim(0, pmax); ax.set_ylim(0, pmax)
    ax.set_xlabel('Actual', fontsize=9, color='#8b92b8')
    ax.set_ylabel('Predicted', fontsize=9, color='#8b92b8')
    ax.tick_params(colors='#8b92b8', labelsize=8)
    ax.set_title(name, fontsize=10, color='white', fontweight='bold', pad=4)
    ax.text(0.97, 0.06, f'R² = {r2:.4f}\nMAE = {mae:.3f}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5, color='white',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#0f1117', alpha=0.8, edgecolor='none'))
    ax.text(0.03, 0.93, cat, transform=ax.transAxes, ha='left', va='top',
            fontsize=7.5, color='white', alpha=0.85)
 
for j in range(n_models, len(axes_flat)):
    axes_flat[j].set_visible(False)
 
legend_elements = [
    Line2D([0],[0], marker='s', color='w', markerfacecolor=v, markersize=9, label=k)
    for k, v in CAT_HEX.items()
] + [Line2D([0],[0], color='#4fc3f7', lw=1.5, ls='--', label='Perfect prediction')]
fig.legend(handles=legend_elements, loc='lower center', ncol=6, fontsize=9,
           frameon=True, facecolor='#1a1d27', edgecolor='#2e3250', labelcolor='white',
           bbox_to_anchor=(0.5, -0.01))
fig.suptitle('Predicted vs Actual — All Models (train_final.csv)',
             fontsize=15, color='white', fontweight='bold', y=1.01)
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('predicted_vs_actual_all_models.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.close()
print('\nSaved → predicted_vs_actual_all_models.png')


Saved → predicted_vs_actual_all_models.png


COMPARING MODELS

In [ ]:
SPINE='#D3D1C7'; GRID='#EBEBEB'; TP='#2C2C2A'; TS='#5F5E5A'; CEILING=0.5793
CAT_COLORS_BAR={'Linear':'#185FA5','Distance':'#534AB7','Tree':'#639922',
                'Ensemble':'#BA7517','Boosting':'#993C1D'}
 
df_res = pd.DataFrame([{'name':k,'r2':v['r2'],'mae':v['mae'],'mse':v['rmse']**2,'cat':v['cat']}
                        for k,v in results_metrics.items()]).sort_values('r2',ascending=False).reset_index(drop=True)
 
fig = plt.figure(figsize=(16, 20)); fig.patch.set_facecolor('#FAFAFA')
gs = GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)
ax_r2=fig.add_subplot(gs[0,:]); ax_mae=fig.add_subplot(gs[1,0])
ax_mse=fig.add_subplot(gs[1,1]); ax_rank=fig.add_subplot(gs[2,0]); ax_scat=fig.add_subplot(gs[2,1])
 
def style_ax(ax, title):
    ax.set_facecolor('#FFFFFF')
    for sp in ax.spines.values(): sp.set_color(SPINE); sp.set_linewidth(0.6)
    ax.tick_params(colors=TS, labelsize=9)
    ax.set_title(title, fontsize=12, fontweight='normal', color=TP, pad=10)
    ax.grid(axis='x', color=GRID, linewidth=0.6, zorder=0); ax.set_axisbelow(True)
 
df_r2s=df_res.sort_values('r2')
bars=ax_r2.barh(df_r2s['name'],df_r2s['r2'],color=df_r2s['cat'].map(CAT_COLORS_BAR),
                height=0.6,edgecolor='white',linewidth=0.5,zorder=3)
ax_r2.axvline(CEILING,color='#E24B4A',linewidth=1.2,linestyle='--',zorder=4)
ax_r2.text(CEILING+0.003,0.3,f'Noise ceiling ({CEILING})',color='#E24B4A',fontsize=8,va='bottom')
ax_r2.set_xlim(0, max(df_res['r2'].max()+0.05, 0.78))
for bar,val in zip(bars,df_r2s['r2']):
    ax_r2.text(val+0.004,bar.get_y()+bar.get_height()/2,f'{val:.4f}',va='center',ha='left',fontsize=8.5,color=TP)
style_ax(ax_r2,'R² score — all models (higher is better)'); ax_r2.set_xlabel('R²',color=TS,fontsize=10)
patches=[mpatches.Patch(color=c,label=cat) for cat,c in CAT_COLORS_BAR.items()]
ax_r2.legend(handles=patches,loc='lower right',fontsize=8.5,framealpha=0.9,edgecolor=SPINE)
 
for ax_m, col, title, xlim in [
    (ax_mae,'mae','MAE — mean absolute error (lower is better)', df_res['mae'].max()*1.2),
    (ax_mse,'mse','MSE — mean squared error (lower is better)',  df_res['mse'].max()*1.2),
]:
    df_s=df_res.sort_values(col,ascending=False)
    bars_m=ax_m.barh(df_s['name'],df_s[col],color=df_s['cat'].map(CAT_COLORS_BAR),
                     height=0.6,edgecolor='white',linewidth=0.5,zorder=3)
    ax_m.set_xlim(0, xlim)
    fmt = '.3f' if col=='mae' else '.1f'
    for bar,val in zip(bars_m,df_s[col]):
        ax_m.text(val*1.01,bar.get_y()+bar.get_height()/2,f'{val:{fmt}}',va='center',ha='left',fontsize=8,color=TP)
    style_ax(ax_m, title); ax_m.set_xlabel(col.upper(),color=TS,fontsize=10)
 
df_rank=df_res.sort_values('r2',ascending=False).reset_index(drop=True)
df_rank['rank']=range(1,len(df_rank)+1)
ax_rank.set_facecolor('#FFFFFF')
for _,row in df_rank.iterrows():
    ax_rank.scatter(row['r2'],row['rank'],color=CAT_COLORS_BAR[row['cat']],s=90,zorder=4)
    ax_rank.text(row['r2']+0.003,row['rank'],row['name'],va='center',fontsize=8.5,color=TP)
ax_rank.axvline(CEILING,color='#E24B4A',linewidth=1.2,linestyle='--',zorder=3)
ax_rank.set_xlim(0.0,max(df_res['r2'].max()+0.06,0.82))
ax_rank.set_ylim(len(df_rank)+0.5,0.5)
ax_rank.set_yticks(df_rank['rank']); ax_rank.set_yticklabels([f'#{r}' for r in df_rank['rank']],fontsize=8.5,color=TS)
ax_rank.set_xlabel('R²',color=TS,fontsize=10)
for sp in ax_rank.spines.values(): sp.set_color(SPINE); sp.set_linewidth(0.6)
ax_rank.grid(axis='x',color=GRID,linewidth=0.6,zorder=0); ax_rank.set_axisbelow(True)
ax_rank.tick_params(colors=TS)
ax_rank.set_title('Ranking by R²',fontsize=12,fontweight='normal',color=TP,pad=10)
 
for _,row in df_res.iterrows():
    ax_scat.scatter(row['r2'],row['mae'],color=CAT_COLORS_BAR[row['cat']],s=80,zorder=4)
    ax_scat.annotate(row['name'],(row['r2'],row['mae']),textcoords='offset points',
                     xytext=(5,2),fontsize=7.5,color=TS)
ax_scat.axvline(CEILING,color='#E24B4A',linewidth=1.2,linestyle='--',zorder=3)
ax_scat.set_xlabel('R²',color=TS,fontsize=10); ax_scat.set_ylabel('MAE',color=TS,fontsize=10)
ax_scat.set_facecolor('#FFFFFF')
for sp in ax_scat.spines.values(): sp.set_color(SPINE); sp.set_linewidth(0.6)
ax_scat.grid(color=GRID,linewidth=0.6,zorder=0); ax_scat.set_axisbelow(True)
ax_scat.tick_params(colors=TS,labelsize=9)
ax_scat.set_title('MAE vs R² (bottom-right = best)',fontsize=12,fontweight='normal',color=TP,pad=10)
fig.suptitle('Model comparison — passenger count prediction (train_final.csv)',
             fontsize=15,fontweight='normal',color=TP,y=0.98)
plt.savefig('model_comparison.png',dpi=150,bbox_inches='tight',facecolor='#FAFAFA')
plt.savefig('model_comparison.pdf',bbox_inches='tight',facecolor='#FAFAFA')
plt.close(); print('Saved → model_comparison.png / .pdf')

Saved → model_comparison.png / .pdf


EPOCH LOG

In [ ]:
if epoch_log:
    n_ep     = len(epoch_log)
    ncols_ep = min(3, n_ep)
    nrows_ep = int(np.ceil(n_ep / ncols_ep))
    fig, axes = plt.subplots(nrows_ep, ncols_ep, figsize=(7*ncols_ep, 4*nrows_ep))
    fig.patch.set_facecolor('#0f1117')
    axes_flat = axes.flatten() if n_ep > 1 else [axes]
 
    for i, (name, log) in enumerate(epoch_log.items()):
        ax = axes_flat[i]
        ax.set_facecolor('#1a1d27')
        for sp in ax.spines.values(): sp.set_edgecolor('#2e3250')
 
        tr  = log.get('train', [])
        va  = log.get('val',   [])
        cps = log.get('checkpoints', None)
        xs  = cps if cps else list(range(1, len(tr)+1))
 
        if tr: ax.plot(xs, tr, color='#4fc3f7', lw=1.5, label='Train RMSE')
        if va: ax.plot(xs, va, color='#ffb74d', lw=1.5, label='Val RMSE')
 
        if va:
            best_idx = int(np.argmin(va))
            best_x   = xs[best_idx]
            ax.axvline(best_x, color='#ce93d8', lw=1, ls='--', alpha=0.8)
            ax.text(best_x + (xs[-1]-xs[0])*0.01, min(va)*1.02,
                    f'Best: {best_x}', color='#ce93d8', fontsize=8)
 
        ax.set_title(name, fontsize=11, color='white', fontweight='bold')
        ax.set_xlabel('Trees / Boosting round', fontsize=9, color='#8b92b8')
        ax.set_ylabel('RMSE', fontsize=9, color='#8b92b8')
        ax.tick_params(colors='#8b92b8', labelsize=8)
        ax.legend(fontsize=8, facecolor='#1a1d27', edgecolor='#2e3250', labelcolor='white')
        ax.grid(color='#2e3250', linewidth=0.5, alpha=0.6)
 
    for j in range(n_ep, len(axes_flat)):
        axes_flat[j].set_visible(False)
 
    fig.suptitle('Learning curves — ensemble & boosting models',
                 fontsize=14, color='white', fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close(); print('Saved → learning_curves.png')

Saved → learning_curves.png


FEATURE IMPORTANCE

In [ ]:
NICE_LABELS = {
    'max_capacity':'Max capacity','month':'Month','day_of_week':'Day of week',
    'week':'Week of year','hour':'Hour','minute':'Minute','is_bus':'Is bus',
    'is_weekend':'Is weekend','travel_from_enc':'Origin (enc)','slot_enc':'Slot (enc)',
    'route_enc':'Route (enc)','day_of_month':'Day of month','quarter':'Quarter',
    'avg_seat_row':'Avg seat row','mpesa_ratio':'Mpesa ratio','time_float':'Time (float)',
    'fuel_price_kes':'Fuel price','fare_kes':'Fare (KES)','distance_km':'Distance (km)',
    'is_holiday':'Is holiday','is_holiday_eve':'Holiday eve','is_holiday_aft':'Holiday after',
    'is_school_hol':'School holiday','days_to_holiday':'Days to holiday',
    'is_rainy_season':'Rainy season','is_long_rains':'Long rains','is_month_end':'Month end',
    'is_month_start':'Month start','is_morning_peak':'Morning peak','is_evening_peak':'Evening peak',
    'n_competitors':'N competitors','origin_pop_proxy':'Origin pop','weekend_holiday':'Weekend×holiday',
    'avg_ticket_lead':'Avg lead days','min_ticket_lead':'Min lead days','max_ticket_lead':'Max lead days',
    'std_ticket_lead':'Std lead days','avg_booking_hour':'Avg booking hour',
    'avg_booking_dow':'Avg booking DOW','n_weekend_bookings':'Weekend bookings',
    'avg_lead_cap_ratio':'Lead/cap ratio','lead_spread':'Lead spread','booking_urgency':'Booking urgency',
    'slot_x_week':'Slot × week','slot_x_dow':'Slot × DOW','slot_x_month':'Slot × month',
    'slot_x_dom':'Slot × DOM','route_x_month':'Route × month','route_x_week':'Route × week',
    'origin_x_slot':'Origin × slot','holiday_x_slot':'Holiday × slot','school_x_slot':'School × slot',
    'fuel_x_dist':'Fuel × dist','comp_x_origin':'Comp × origin','rain_x_slot':'Rain × slot',
    'fare_x_dist':'Fare / dist','capacity_x_bus':'Cap × bus','lead_x_slot':'Lead × slot',
    'lead_x_week':'Lead × week','lead_x_holiday':'Lead × holiday','lead_x_school':'Lead × school',
    'spread_x_slot':'Spread × slot','urgency_x_origin':'Urgency × origin','minlead_x_slot':'MinLead × slot',
    'slot_mean':'Slot mean','route_mean':'Route mean','origin_mean':'Origin mean',
    'dow_mean':'DOW mean','month_mean':'Month mean','hour_mean':'Hour mean',
    'week_mean':'Week mean','quarter_mean':'Quarter mean','dom_mean':'DOM mean',
}
 
CAT_COLORS_FEAT = {'raw':'#4fc3f7','mean_enc':'#ffb74d','booking':'#a5d6a7','interaction':'#ce93d8'}
 
def feat_color(f):
    if any(x in f for x in ['lead','booking','urgency','spread','weekend_book']): return CAT_COLORS_FEAT['booking']
    if 'x' in f:    return CAT_COLORS_FEAT['interaction']
    if 'mean' in f: return CAT_COLORS_FEAT['mean_enc']
    return CAT_COLORS_FEAT['raw']
 
tree_fi_models = {
    'Decision Tree':  DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest':  RandomForestRegressor(n_estimators=300, max_depth=15, n_jobs=-1, random_state=42),
    'Extra Trees':    ExtraTreesRegressor(n_estimators=300, max_depth=15, n_jobs=-1, random_state=42),
    'GBM':            GradientBoostingRegressor(n_estimators=300, learning_rate=0.012,
                                                max_depth=6, subsample=0.8, random_state=42),
    'AdaBoost':       AdaBoostRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
    'Bagging':        BaggingRegressor(n_estimators=100, random_state=42),
}
if XGBOOST_OK:
    tree_fi_models['XGBoost'] = XGBRegressor(n_estimators=500, learning_rate=0.03,
        max_depth=6, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42, verbosity=0)
if LGBM_OK:
    tree_fi_models['LightGBM'] = LGBMRegressor(n_estimators=500, learning_rate=0.03,
        num_leaves=63, max_depth=8, n_jobs=-1, random_state=42, verbose=-1)
if CATBOOST_OK:
    tree_fi_models['CatBoost'] = CatBoostRegressor(iterations=500, learning_rate=0.03,
        depth=6, l2_leaf_reg=3.0, random_seed=42, verbose=0)
 
linear_fi_models = {
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(alpha=1.0),
    'Lasso':             Lasso(alpha=0.1),
    'ElasticNet':        ElasticNet(alpha=0.1, l1_ratio=0.5),
}
 
fi_results = {}; perm_results = {}
print('\nFitting models for feature importance...')
 
for name, model in tree_fi_models.items():
    try:
        if name == 'XGBoost':
            model.fit(X_train[FINAL_FEATURES], y_train,
                      eval_set=[(X_test[FINAL_FEATURES], y_test)], verbose=False)
        elif name == 'CatBoost':
            model.fit(X_train[FINAL_FEATURES], y_train,
                      eval_set=(X_test[FINAL_FEATURES], y_test), verbose=False)
        else:
            model.fit(X_train[FINAL_FEATURES], y_train)
        fi = (np.mean([e.feature_importances_ for e in model.estimators_], axis=0)
              if name == 'Bagging' else model.feature_importances_)
        fi_results[name] = pd.Series(fi, index=FINAL_FEATURES)
        print(f'  {name:<22}  R²={r2_score(y_test, model.predict(X_test[FINAL_FEATURES])):.4f}')
    except Exception as e:
        print(f'  {name} ERROR: {e}')
 
for name, model in linear_fi_models.items():
    model.fit(X_train[FINAL_FEATURES], y_train)
    perm = permutation_importance(model, X_test[FINAL_FEATURES], y_test,
                                  n_repeats=10, random_state=42, n_jobs=-1)
    perm_results[name] = pd.Series(perm.importances_mean, index=FINAL_FEATURES)
 
BG_FI='#0f1117'; CARD='#1a1d27'; GRID_FI='#2e3250'; TXT='#c2c0b6'; SUB='#8b92b8'
all_names_fi = list(fi_results.keys()) + list(perm_results.keys())
n_fi=len(all_names_fi); ncols_fi=3; nrows_fi=int(np.ceil(n_fi/ncols_fi))
 
fig, axes = plt.subplots(nrows_fi, ncols_fi, figsize=(21, nrows_fi * 5.2))
fig.patch.set_facecolor(BG_FI)
axes_flat = axes.flatten()
 
def plot_fi(ax, series, name, kind='native'):
    s = series.sort_values(ascending=True)
    labels=[NICE_LABELS.get(f, f) for f in s.index]; vals=s.values
    colors=[feat_color(f) for f in s.index]
    vmax=vals.max(); vp=(vals/vmax*100) if vmax>0 else vals
    bars=ax.barh(labels, vp, color=colors, edgecolor='none', height=0.65)
    for bar, v in zip(bars, vp):
        if v > 0.5:
            ax.text(min(v+1.5,97), bar.get_y()+bar.get_height()/2,
                    f'{v:.1f}', va='center', ha='left', fontsize=7.5, color=TXT)
    ax.set_facecolor(CARD)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_FI)
    ax.tick_params(colors=TXT, labelsize=8.5)
    ax.set_xlabel('Relative importance (%)', fontsize=8.5, color=SUB)
    ax.set_xlim(0, 110)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0f}%'))
    ax.grid(axis='x', color=GRID_FI, linewidth=0.5, alpha=0.6); ax.set_axisbelow(True)
    ax.set_title(name, fontsize=11, color='white', fontweight='bold', pad=6)
    tag = 'Permutation importance' if kind=='perm' else 'Feature importance'
    ax.text(1, 1.01, tag, transform=ax.transAxes, ha='right', va='bottom', fontsize=7.5, color=SUB)
 
for i, name in enumerate(all_names_fi):
    ax = axes_flat[i]
    if name in fi_results: plot_fi(ax, fi_results[name], name, kind='native')
    else:                   plot_fi(ax, perm_results[name], name, kind='perm')
 
for j in range(n_fi, len(axes_flat)): axes_flat[j].set_visible(False)
 
legend_fi = [
    Patch(facecolor=CAT_COLORS_FEAT['raw'],         label='Raw / schedule features'),
    Patch(facecolor=CAT_COLORS_FEAT['mean_enc'],    label='Mean encodings'),
    Patch(facecolor=CAT_COLORS_FEAT['booking'],     label='Booking features'),
    Patch(facecolor=CAT_COLORS_FEAT['interaction'], label='Interaction features'),
]
fig.legend(handles=legend_fi, loc='lower center', ncol=4, fontsize=10,
           frameon=True, facecolor=CARD, edgecolor=GRID_FI, labelcolor=TXT,
           bbox_to_anchor=(0.5, -0.015))
fig.suptitle('Feature importance — All Models (train_final.csv)',
             fontsize=16, color='white', fontweight='bold', y=1.01)
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('feature_importance_all_models.png', dpi=150, bbox_inches='tight', facecolor=BG_FI)
plt.close(); print('Saved → feature_importance_all_models.png')


Fitting models for feature importance...
  Decision Tree           R²=0.9038
  Random Forest           R²=0.9510
  Extra Trees             R²=0.9583
  GBM                     R²=0.9571
  AdaBoost                R²=0.9006
  Bagging                 R²=0.9505
  XGBoost                 R²=0.9661
  LightGBM                R²=0.9668
  CatBoost                R²=0.9625
Saved → feature_importance_all_models.png
